# 09. Explainability

## 0. Setup

In [ ]:
import csv
import random
import re
import json
import time
import math
import copy
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[WARN] shap not installed -- Part 2 will fail loudly when reached. pip install shap.")

MANIFEST = Path('~/Desktop/convscript/Code/Manifest/manifests/master_manifest.csv').expanduser()
ROOT     = Path('~/Desktop/Thesis/TR-6').expanduser()
GAS_NORM = Path('~/Desktop/convscript/Code/Manifest/GasNorm/gas_norm_stats.json').expanduser()
RUNS     = Path('~/Desktop/convscript/runs').expanduser()

DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {DEVICE}')

FUSION_OUT = RUNS / "04_fusion_model"     
ABLATION_OUT = RUNS / "05_ablation_study" 
EXPLAIN_OUT = RUNS / "09_explainability"
EXPLAIN_OUT.mkdir(parents=True, exist_ok=True)

RUN_KEY = "model_a_winner"
FAILURES = []


## Data

In [ ]:
FRUIT_LIST   = ['Banana', 'Carrot', 'Guava', 'Indian_Gooseberry', 'Mango', 'Tomato']
FRUIT_TO_IDX = {f: i for i, f in enumerate(FRUIT_LIST)}
SESSION_TO_IDX = {'morning': 0, 'afternoon': 1, 'evening': 2}
LABEL_TO_IDX   = {'not_spoiled': 0, 'spoiled': 1}
IMAGE_SIZE     = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
SESSION_HOUR_BUCKETS = {'morning': (5, 11), 'afternoon': (12, 16), 'evening': (16, 19)}
IR_PATTERN   = re.compile(r'(\d{8})_(\d{6})_([\d.]+)C_([\d.]+)C\.jpg$', re.IGNORECASE)
SRGB_PATTERN = re.compile(r'^(\d{8})_(\d{6})[^/]*\.jpg$', re.IGNORECASE)
NUM_FRUITS   = len(FRUIT_TO_IDX)
THRESHOLDS_TO_SWEEP = [t / 100 for t in range(10, 91)]


def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default


def hour_to_session(hour):
    for name, (lo, hi) in SESSION_HOUR_BUCKETS.items():
        if lo <= hour < hi:
            return name
    return 'unknown'


def find_ir_folder(base):
    for name in ['IR_fusion_images', 'IR_Fusion_images', 'ir_fusion_images']:
        p = base / name
        if p.exists():
            return p
    return base / 'IR_fusion_images'


def group_images_by_session(folder, pattern):
    result = defaultdict(lambda: defaultdict(list))
    if not folder.exists():
        return {}
    for f in sorted(folder.iterdir()):
        m = pattern.search(f.name)
        if not m:
            continue
        session = hour_to_session(int(m.group(2)[:2]))
        if session != 'unknown':
            result[m.group(1)][session].append(f)
    return dict(result)


def load_image(path, transform):
    return transform(Image.open(path).convert('RGB'))


def get_rgb_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_ir_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


In [ ]:
class TrimodalDataset(Dataset):
    def __init__(self, manifest_path, root, train=True, modality_dropout_prob=0.3,
                 exclude_flagged=True, split=None, gas_norm_stats_path=None,
                 cache_images=True, shared_pixel_cache=None, gas_only=False):
        self.root = Path(root); self.train = train; self.modality_dropout_prob = modality_dropout_prob
        self.rgb_transform = get_rgb_transform(train); self.ir_transform = get_ir_transform(train)
        self.gas_norm_stats = None
        if gas_norm_stats_path:
            with open(gas_norm_stats_path) as f:
                self.gas_norm_stats = json.load(f)['stats']
        with open(manifest_path, newline='') as f:
            all_rows = list(csv.DictReader(f))
        if exclude_flagged:
            all_rows = [r for r in all_rows if r.get('exclude', '').strip().lower() != 'true']
        if split:
            all_rows = [r for r in all_rows if r.get('split', '').strip().lower() == split.lower()]
        self.samples = []; self._build_samples(all_rows)
        self.gas_only = gas_only; self._image_cache = {}
        if not gas_only:
            self._build_image_index()
        self._pixel_cache = {}
        if not gas_only:
            if shared_pixel_cache is not None:
                self._pixel_cache = shared_pixel_cache
            elif cache_images:
                self._warmup_pixel_cache()
        print(f'TrimodalDataset: {len(self.samples)} sessions (split={split})')

    def _build_samples(self, rows):
        for row in rows:
            fruit = row['fruit']; label = row['label']; date_str = row['actual_date']
            global_day = int(row['corrected_day_index'])
            days_until = safe_float(row.get('days_until_spoilage', ''), default=-1.0)
            for session in ['morning', 'afternoon', 'evening']:
                si = SESSION_TO_IDX[session]
                ir_avail = safe_float(row.get(f'ir_{session}_count', 0)) > 0
                ir_tmin = safe_float(row.get(f'ir_{session}_tmin', ''), 0.0)
                ir_tmax = safe_float(row.get(f'ir_{session}_tmax', ''), 0.0)
                ir_trange = safe_float(row.get(f'ir_{session}_trange', ''), 0.0)
                srgb_avail = safe_float(row.get(f'srgb_{session}_count', 0)) > 0
                gas_avail = row.get(f'methane_{session}_present', '').strip().upper() == 'TRUE'
                mean_ppm = safe_float(row.get(f'methane_{session}_ppm', ''), 0.0)
                std_ppm = safe_float(row.get(f'methane_{session}_std', ''), 0.0)
                left_ppm = safe_float(row.get(f'methane_{session}_left', ''), 0.0)
                right_ppm = safe_float(row.get(f'methane_{session}_right', ''), 0.0)
                if not ir_avail and not srgb_avail and not gas_avail:
                    continue
                self.samples.append({'fruit': fruit, 'label': label, 'actual_date': date_str,
                    'corrected_day_index': global_day, 'session': session, 'session_idx': si,
                    'fruit_idx': FRUIT_TO_IDX.get(fruit, 0), 'label_idx': LABEL_TO_IDX.get(label, 0),
                    'days_until_spoilage': days_until, 'ir_tmin': ir_tmin, 'ir_tmax': ir_tmax,
                    'ir_trange': ir_trange, 'gas_mean': mean_ppm, 'gas_std': std_ppm,
                    'gas_left': left_ppm, 'gas_right': right_ppm, 'gas_asym': abs(left_ppm - right_ppm),
                    'rgb_available': srgb_avail, 'ir_available': ir_avail, 'gas_available': gas_avail})

    def _build_image_index(self):
        label_map = {'not_spoiled': 'Not_spoiled', 'spoiled': 'Spoiled'}
        for fruit in FRUIT_LIST:
            for lk, lf in label_map.items():
                base = self.root / 'Classified' / fruit / lf
                for d, ss in group_images_by_session(base / 'sRGB_images', SRGB_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['rgb'] = ps
                for d, ss in group_images_by_session(find_ir_folder(base), IR_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['ir'] = ps
        banana_ir = group_images_by_session(find_ir_folder(self.root / 'Normal' / 'Banana'), IR_PATTERN)
        spoiled_dates = {date for (fr, lb, date, _) in self._image_cache if fr == 'Banana' and lb == 'spoiled'}
        for d, ss in banana_ir.items():
            if d in spoiled_dates:
                for s, ps in ss.items():
                    key = ('Banana', 'spoiled', d, s)
                    self._image_cache.setdefault(key, {'rgb': [], 'ir': []})
                    if not self._image_cache[key].get('ir'):
                        self._image_cache[key]['ir'] = ps

    def _load_cached(self, path, transform):
        cached = self._pixel_cache.get(str(path))
        if cached is not None:
            return transform(Image.fromarray(cached.permute(1, 2, 0).numpy()))
        return load_image(path, transform)

    def _warmup_pixel_cache(self):
        all_paths = set()
        for v in self._image_cache.values():
            all_paths.update(v.get('rgb', [])); all_paths.update(v.get('ir', []))
        for path in all_paths:
            try:
                img = self.rgb_transform.transforms[0](Image.open(path).convert('RGB'))
                self._pixel_cache[str(path)] = torch.from_numpy(np.array(img)).permute(2, 0, 1)
            except Exception:
                pass

    def _normalize_gas(self, fruit, session, mean_ppm, std_ppm, left_ppm, right_ppm):
        if self.gas_norm_stats is None:
            return mean_ppm, std_ppm, left_ppm, right_ppm
        s = self.gas_norm_stats.get(fruit, {}).get(session, {})

        def norm(val, key):
            st = s.get(key, {'mean': 0., 'std': 1.})
            return (val - st['mean']) / st['std']
        return norm(mean_ppm, 'mean_ppm'), norm(std_ppm, 'std_ppm'), norm(left_ppm, 'left_ppm'), norm(right_ppm, 'right_ppm')

    def _apply_dropout(self, rgb_avail, ir_avail, gas_avail):
        if not self.train:
            return rgb_avail, ir_avail, gas_avail
        while True:
            r = rgb_avail and (random.random() > self.modality_dropout_prob)
            i = ir_avail and (random.random() > self.modality_dropout_prob)
            g = gas_avail and (random.random() > self.modality_dropout_prob)
            if r or i or g:
                return r, i, g

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fruit = s['fruit']; label = s['label']; date_str = s['actual_date']
        session = s['session']; si = s['session_idx']
        rgb_avail, ir_avail, gas_avail = self._apply_dropout(s['rgb_available'], s['ir_available'], s['gas_available'])
        cached = self._image_cache.get((fruit, label, date_str, session), {'rgb': [], 'ir': []})
        if not self.gas_only and rgb_avail and cached['rgb']:
            rgb_tensors = torch.stack([self._load_cached(p, self.rgb_transform) for p in cached['rgb']])
        else:
            rgb_avail = False; rgb_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        if not self.gas_only and ir_avail and cached['ir']:
            ir_tensors = torch.stack([self._load_cached(p, self.ir_transform) for p in cached['ir']])
        else:
            ir_avail = False; ir_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        ir_scalars = torch.tensor([s['ir_tmin'], s['ir_tmax'], s['ir_trange']], dtype=torch.float32) if ir_avail else torch.zeros(3)
        if gas_avail:
            m, sd, l, r = self._normalize_gas(fruit, session, s['gas_mean'], s['gas_std'], s['gas_left'], s['gas_right'])
            gas = torch.tensor([m, sd, l, r, abs(l - r), float(si) / 2.], dtype=torch.float32)
        else:
            gas = torch.zeros(6)
        return {'rgb_images': rgb_tensors, 'ir_images': ir_tensors, 'ir_scalars': ir_scalars, 'gas': gas,
                'rgb_available': torch.tensor(rgb_avail, dtype=torch.bool),
                'ir_available': torch.tensor(ir_avail, dtype=torch.bool),
                'gas_available': torch.tensor(gas_avail, dtype=torch.bool),
                'fruit_idx': torch.tensor(s['fruit_idx'], dtype=torch.long),
                'session_idx': torch.tensor(si, dtype=torch.long),
                'label': torch.tensor(s['label_idx'], dtype=torch.long),
                'days_until_spoilage': torch.tensor(s['days_until_spoilage'], dtype=torch.float32),
                'fruit': fruit, 'actual_date': date_str, 'corrected_day_index': s['corrected_day_index']}


In [ ]:
class DayLevelSequenceDataset(Dataset):
    def __init__(self, manifest_path=MANIFEST, root=ROOT, split="train",
                 gas_norm_stats_path=GAS_NORM, shared_pixel_cache=None):
        base = TrimodalDataset(
            manifest_path, root, train=False, modality_dropout_prob=0.0,
            split=split, gas_norm_stats_path=gas_norm_stats_path, cache_images=True,
            shared_pixel_cache=shared_pixel_cache,
        )
        self._pixel_cache = base._pixel_cache
        day_groups = defaultdict(list)
        for i in range(len(base)):
            item = base[i]
            key = (item["fruit"], int(item["label"].item()), int(item["corrected_day_index"]))
            day_groups[key].append(item)
        day_entries = {}
        for (fruit, label, day_idx), sessions in day_groups.items():
            sessions.sort(key=lambda s: int(s["session_idx"].item()))
            rgb_img, ir_img = None, None
            for s in sessions:
                if rgb_img is None and bool(s["rgb_available"].item()) and len(s["rgb_images"]) > 0:
                    rgb_img = s["rgb_images"][0]
                if ir_img is None and bool(s["ir_available"].item()) and len(s["ir_images"]) > 0:
                    ir_img = s["ir_images"][0]
            gas_vals = [s["gas"] for s in sessions if bool(s["gas_available"].item())]
            gas_avail = len(gas_vals) > 0
            gas_vec = torch.stack(gas_vals).mean(dim=0) if gas_avail else torch.zeros(6)
            day_entries.setdefault((fruit, label), []).append({
                "day_idx": day_idx, "rgb_img": rgb_img, "ir_img": ir_img,
                "gas_vec": gas_vec, "gas_avail": gas_avail,
                "fruit_idx": sessions[0]["fruit_idx"], "label": sessions[0]["label"],
                "days_until": sessions[0]["days_until_spoilage"],
            })
        self.trajectories = []
        for (fruit, label), days in day_entries.items():
            days.sort(key=lambda d: d["day_idx"])
            last_observed_idx = None
            for d in days:
                d["delta"] = 0.0 if last_observed_idx is None else float(d["day_idx"] - last_observed_idx)
                if d["gas_avail"]:
                    last_observed_idx = d["day_idx"]
            self.trajectories.append({"fruit": fruit, "label": label, "days": days})
        n_days_total = sum(len(t["days"]) for t in self.trajectories)
        print(f"  DayLevelSequenceDataset: {len(self.trajectories)} trajectories, {n_days_total} day-entries")

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        traj = self.trajectories[idx]["days"]
        fruit = self.trajectories[idx]["fruit"]
        label_str = self.trajectories[idx]["label"]
        T = len(traj)
        blank_rgb = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        blank_ir = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        rgb_seq = torch.stack([d["rgb_img"] if d["rgb_img"] is not None else blank_rgb for d in traj])
        ir_seq = torch.stack([d["ir_img"] if d["ir_img"] is not None else blank_ir for d in traj])
        rgb_avail = torch.tensor([d["rgb_img"] is not None for d in traj], dtype=torch.bool)
        ir_avail = torch.tensor([d["ir_img"] is not None for d in traj], dtype=torch.bool)
        gas_seq = torch.stack([d["gas_vec"] for d in traj])
        gas_mask = torch.tensor([d["gas_avail"] for d in traj], dtype=torch.float32)
        gas_delta = torch.tensor([d["delta"] for d in traj], dtype=torch.float32)
        return {
            "rgb_seq": rgb_seq, "ir_seq": ir_seq, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
            "gas_seq": gas_seq, "gas_mask": gas_mask, "gas_delta": gas_delta, "T": T,
            "fruit_idx": traj[0]["fruit_idx"], "label": traj[-1]["label"],
            "days_until": traj[-1]["days_until"], "fruit": fruit, "label_str": label_str,
        }


def day_sequence_collate(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)
    rgb = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    ir = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    rgb_avail = torch.zeros(B, max_T, dtype=torch.bool)
    ir_avail = torch.zeros(B, max_T, dtype=torch.bool)
    gas = torch.zeros(B, max_T, 6)
    gas_mask = torch.zeros(B, max_T)
    gas_delta = torch.zeros(B, max_T)
    pad_mask = torch.ones(B, max_T, dtype=torch.bool)
    fruit_idx = torch.zeros(B, dtype=torch.long)
    label = torch.zeros(B, dtype=torch.long)
    days_until = torch.zeros(B, dtype=torch.float32)
    fruits, label_strs = [], []
    for i, b in enumerate(batch):
        T = b["T"]
        rgb[i, :T] = b["rgb_seq"]; ir[i, :T] = b["ir_seq"]
        rgb_avail[i, :T] = b["rgb_avail"]; ir_avail[i, :T] = b["ir_avail"]
        gas[i, :T] = b["gas_seq"]; gas_mask[i, :T] = b["gas_mask"]; gas_delta[i, :T] = b["gas_delta"]
        pad_mask[i, :T] = False
        fruit_idx[i] = b["fruit_idx"]; label[i] = b["label"]; days_until[i] = b["days_until"]
        fruits.append(b.get("fruit")); label_strs.append(b.get("label_str"))
    return {"rgb_seq": rgb, "ir_seq": ir, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
            "gas_seq": gas, "gas_mask": gas_mask, "gas_delta": gas_delta, "pad_mask": pad_mask,
            "fruit_idx": fruit_idx, "label": label, "days_until": days_until,
            "fruit": fruits, "label_str": label_strs}


## Model building blocks

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, channels, 1, bias=False))

    def forward(self, x):
        return torch.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out, _ = x.max(dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = x * self.channel_attn(x)
        sa = self.spatial_attn(x)
        return x * sa, sa


BACKBONE_FEATURE_DIMS = {"efficientnet_b0": 1280}


def _infer_feature_dim(backbone, img_size=IMAGE_SIZE):
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(1, 3, img_size, img_size)
        out = backbone(dummy)
    return out.shape[1]


def build_cnn_backbone(name: str):
    if name == "efficientnet_b0":
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        return base.features, BACKBONE_FEATURE_DIMS[name]
    elif name == "convnext_tiny":
        base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        backbone = base.features
        feat_dim = _infer_feature_dim(backbone)
        BACKBONE_FEATURE_DIMS[name] = feat_dim
        return backbone, feat_dim
    else:
        raise ValueError(f"Unknown backbone: {name}")


class VisualEncoderToggle(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", freeze_backbone=True,
                 use_cbam=True, chunk_size=8):
        super().__init__()
        self.use_cbam = use_cbam
        self.backbone_name = backbone_name
        self.chunk_size = chunk_size
        self.backbone, self.feature_dim = build_cnn_backbone(backbone_name)
        if use_cbam:
            self.cbam = CBAM(self.feature_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if freeze_backbone:
            self.freeze_backbone()

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def unfreeze_last_n_layers(self, n: int):
        self.freeze_backbone()
        for child in list(self.backbone.children())[-n:]:
            for p in child.parameters():
                p.requires_grad = True

    def forward(self, seq: torch.Tensor):
        B, T, C, H, W = seq.shape
        flat = seq.reshape(B * T, C, H, W)
        feats_list = []
        needs_ckpt = (torch.is_grad_enabled()
                      and any(p.requires_grad for p in self.backbone.parameters()))
        for start in range(0, flat.shape[0], self.chunk_size):
            chunk = flat[start:start + self.chunk_size]
            if needs_ckpt:
                feat_map = checkpoint(self.backbone, chunk, use_reentrant=False)
            else:
                feat_map = self.backbone(chunk)
            if self.use_cbam:
                feat_map, _ = self.cbam(feat_map)
            feats_list.append(self.pool(feat_map).flatten(1))
        feats = torch.cat(feats_list, dim=0).view(B, T, -1)
        return feats


class GRUDCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, x_mean=None):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        if x_mean is None:
            x_mean = [0.0] * input_dim
        self.register_buffer("x_mean", torch.as_tensor(x_mean, dtype=torch.float32))
        self.W_gamma_x = nn.Linear(1, input_dim)
        self.W_gamma_h = nn.Linear(1, hidden_dim)
        self.gru_cell = nn.GRUCell(input_dim * 2, hidden_dim)

    def forward(self, x_t, m_t, delta_t, x_last, h_prev):
        gamma_x = torch.exp(-torch.clamp(self.W_gamma_x(delta_t), min=0.0))
        x_bar = self.x_mean.unsqueeze(0).expand_as(x_t)
        x_hat = m_t * x_t + (1 - m_t) * (gamma_x * x_last + (1 - gamma_x) * x_bar)
        gamma_h = torch.exp(-torch.clamp(self.W_gamma_h(delta_t), min=0.0))
        h_t = self.gru_cell(torch.cat([x_hat, m_t], dim=-1), gamma_h * h_prev)
        return h_t, x_hat


class GasGRUDSequenceEncoder(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, x_mean=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = GRUDCell(input_dim, hidden_dim, x_mean)

    def forward(self, gas_seq, gas_mask, gas_delta, pad_mask):
        B, T, D = gas_seq.shape
        device = gas_seq.device
        h = torch.zeros(B, self.hidden_dim, device=device)
        x_last = torch.zeros(B, D, device=device)
        h_seq = []
        for t in range(T):
            x_t = gas_seq[:, t]
            m_t = gas_mask[:, t].unsqueeze(-1).expand(-1, D)
            delta_t = gas_delta[:, t].unsqueeze(-1)
            valid_t = (~pad_mask[:, t]).float().unsqueeze(-1)
            h_new, x_hat = self.cell(x_t, m_t, delta_t, x_last, h)
            h = valid_t * h_new + (1 - valid_t) * h
            x_last = torch.where(m_t.bool(), x_t, x_last)
            h_seq.append(h)
        return torch.stack(h_seq, dim=1)


class GasEncoderMeanImpute(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gru = nn.GRU(input_dim + 1, hidden_dim, batch_first=True)

    def forward(self, gas_seq, gas_mask, gas_delta, pad_mask):
        imputed = gas_seq * gas_mask.unsqueeze(-1)
        gru_input = torch.cat([imputed, gas_mask.unsqueeze(-1)], dim=-1)
        lengths = (~pad_mask).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(gru_input, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=gas_seq.shape[1])
        return out


class CrossModalFusion(nn.Module):
    def __init__(self, d_model=64, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads,
                                           dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, rgb_t, ir_t, gas_t, rgb_avail_t, ir_avail_t, gas_avail_t):
        tokens = torch.stack([rgb_t, ir_t, gas_t], dim=1)
        avail = torch.stack([rgb_avail_t, ir_avail_t, gas_avail_t], dim=1)
        key_padding_mask = ~avail
        fully_missing = key_padding_mask.all(dim=1)
        if fully_missing.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[fully_missing] = False
        attended, attn_w = self.attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask,
                                      need_weights=True, average_attn_weights=True)
        out = self.norm(tokens + attended)
        return out.mean(dim=1), attn_w


class FusionToggle(nn.Module):
    def __init__(self, d_model=64, num_heads=4, dropout=0.1, use_cross_attn=True):
        super().__init__()
        self.use_cross_attn = use_cross_attn
        if use_cross_attn:
            self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads,
                                               dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_model)
        else:
            self.concat_proj = nn.Sequential(
                nn.Linear(d_model * 3, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout),
            )

    def forward(self, rgb_t, ir_t, gas_t, rgb_avail_t, ir_avail_t, gas_avail_t):
        if self.use_cross_attn:
            tokens = torch.stack([rgb_t, ir_t, gas_t], dim=1)
            avail = torch.stack([rgb_avail_t, ir_avail_t, gas_avail_t], dim=1)
            key_padding_mask = ~avail
            fully_missing = key_padding_mask.all(dim=1)
            if fully_missing.any():
                key_padding_mask = key_padding_mask.clone()
                key_padding_mask[fully_missing] = False
            attended, attn_w = self.attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask,
                                          need_weights=True, average_attn_weights=True)
            out = self.norm(tokens + attended)
            return out.mean(dim=1), attn_w
        else:
            concat = torch.cat([rgb_t, ir_t, gas_t], dim=-1)
            return self.concat_proj(concat), None


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, dim_feedforward=512,
                 dropout=0.1, max_len=100):
        super().__init__()
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, z_seq: torch.Tensor, pad_mask: torch.Tensor):
        z_seq = self.pos_enc(z_seq)
        return self.encoder(z_seq, src_key_padding_mask=pad_mask)


class LSTMTemporalModule(nn.Module):
    def __init__(self, d_model=64, hidden_dim=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_model, hidden_dim, num_layers=num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.out_proj = nn.Linear(hidden_dim, d_model) if hidden_dim != d_model else nn.Identity()

    def forward(self, z_seq: torch.Tensor, pad_mask: torch.Tensor):
        lengths = (~pad_mask).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(z_seq, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=z_seq.shape[1])
        return self.out_proj(out)


def gather_last_valid(H: torch.Tensor, pad_mask: torch.Tensor):
    device = H.device
    lengths = (~pad_mask).sum(dim=1)
    last_idx = (lengths - 1).clamp(min=0)
    B = H.shape[0]
    return H[torch.arange(B, device=device), last_idx], last_idx


class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, n_tasks: int = 3):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, losses: list):
        total = 0.0
        for i, L_i in enumerate(losses):
            precision = torch.exp(-2 * self.log_sigma[i])
            total = total + 0.5 * precision * L_i + self.log_sigma[i]
        return total


## Full Model class, Ablation model family

In [ ]:
class TrimodalFusionModel(nn.Module):
    def __init__(self, rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0",
                 d_model=64, gas_hidden=64, num_heads=4, temporal_layers=2,
                 dropout=0.5, cls_dropout=0.5, freeze_visual_backbone=True):
        super().__init__()
        self.d_model = d_model
        self.rgb_encoder = VisualEncoderToggle(rgb_backbone_name, freeze_visual_backbone, use_cbam=True)
        self.ir_encoder = VisualEncoderToggle(ir_backbone_name, freeze_visual_backbone, use_cbam=True)
        self.gas_encoder = GasGRUDSequenceEncoder(input_dim=6, hidden_dim=gas_hidden)

        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model),
                                       nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.ir_proj = nn.Sequential(nn.Linear(self.ir_encoder.feature_dim, d_model),
                                      nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.gas_proj = nn.Sequential(nn.Linear(gas_hidden, d_model),
                                       nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

        self.fusion = CrossModalFusion(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.temporal = TemporalTransformer(d_model=d_model, nhead=num_heads,
                                             num_layers=temporal_layers, dropout=dropout)

        self.cls_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(),
                                       nn.Dropout(cls_dropout), nn.Linear(128, 2))
        self.reg_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(),
                                       nn.Dropout(cls_dropout), nn.Linear(128, 1), nn.ReLU())
        self.gas_recon_head = nn.Sequential(nn.Linear(d_model * 2, 128), nn.ReLU(),
                                             nn.Dropout(cls_dropout), nn.Linear(128, 6))

    def forward(self, batch: dict):
        device = next(self.parameters()).device
        rgb_seq = batch["rgb_seq"].to(device); ir_seq = batch["ir_seq"].to(device)
        rgb_avail = batch["rgb_avail"].to(device); ir_avail = batch["ir_avail"].to(device)
        gas_seq = batch["gas_seq"].to(device); gas_mask = batch["gas_mask"].to(device)
        gas_delta = batch["gas_delta"].to(device); pad_mask = batch["pad_mask"].to(device)
        B, T = rgb_avail.shape

        rgb_feat = self.rgb_encoder(rgb_seq)
        ir_feat = self.ir_encoder(ir_seq)
        gas_feat = self.gas_encoder(gas_seq, gas_mask, gas_delta, pad_mask)

        rgb_proj = self.rgb_proj(rgb_feat) * rgb_avail.unsqueeze(-1).float()
        ir_proj = self.ir_proj(ir_feat) * ir_avail.unsqueeze(-1).float()
        gas_proj = self.gas_proj(gas_feat) * gas_mask.unsqueeze(-1)

        rgb_flat = rgb_proj.reshape(B * T, -1)
        ir_flat = ir_proj.reshape(B * T, -1)
        gas_flat = gas_proj.reshape(B * T, -1)
        rgb_av_flat = rgb_avail.reshape(B * T)
        ir_av_flat = ir_avail.reshape(B * T)
        gas_av_flat = gas_mask.reshape(B * T).bool()
        z_flat, _ = self.fusion(rgb_flat, ir_flat, gas_flat, rgb_av_flat, ir_av_flat, gas_av_flat)
        z_seq = z_flat.reshape(B, T, -1)

        H = self.temporal(z_seq, pad_mask)
        H_T, last_idx = gather_last_valid(H, pad_mask)
        cls_logits = self.cls_head(H_T)
        reg_output = self.reg_head(H_T)

        rgb_last = rgb_proj[torch.arange(B, device=device), last_idx]
        ir_last = ir_proj[torch.arange(B, device=device), last_idx]
        gas_recon = self.gas_recon_head(torch.cat([rgb_last, ir_last], dim=1))

        return {"cls_logits": cls_logits, "reg_output": reg_output, "gas_recon": gas_recon, "last_idx": last_idx}


ABLATION_CONFIGS = {
    "ablation_1_no_cbam": {
        "fusion_type": "intermediate", "use_cbam": False, "use_cross_attn": True,
        "temporal_type": "transformer", "missing_handling": "grud",
    },
    "ablation_4_mean_impute_not_grud": {
        "fusion_type": "intermediate", "use_cbam": True, "use_cross_attn": True,
        "temporal_type": "transformer", "missing_handling": "mean",
    },
}


class AblationModel(nn.Module):
    def __init__(self, config: dict, d_model=64, backbone_name="efficientnet_b0", gas_hidden=64,
                 num_heads=4, temporal_layers=2, dropout=0.5, cls_dropout=0.5,
                 freeze_visual_backbone=True):
        super().__init__()
        self.config = config
        self.d_model = d_model
        self.fusion_type = config["fusion_type"]

        self.rgb_encoder = VisualEncoderToggle(backbone_name, freeze_visual_backbone, use_cbam=config["use_cbam"])
        self.ir_encoder = VisualEncoderToggle(backbone_name, freeze_visual_backbone, use_cbam=config["use_cbam"])
        if config["missing_handling"] == "grud":
            self.gas_encoder = GasGRUDSequenceEncoder(input_dim=6, hidden_dim=gas_hidden)
        else:
            self.gas_encoder = GasEncoderMeanImpute(input_dim=6, hidden_dim=gas_hidden)

        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model),
                                       nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.ir_proj = nn.Sequential(nn.Linear(self.ir_encoder.feature_dim, d_model),
                                      nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.gas_proj = nn.Sequential(nn.Linear(gas_hidden, d_model),
                                       nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

        def make_temporal():
            if config["temporal_type"] == "transformer":
                return TemporalTransformer(d_model=d_model, nhead=num_heads,
                                            num_layers=temporal_layers, dropout=dropout)
            return LSTMTemporalModule(d_model=d_model, hidden_dim=d_model, dropout=dropout)

        def make_heads():
            cls = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 2))
            reg = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout),
                                 nn.Linear(128, 1), nn.ReLU())
            return cls, reg

        assert self.fusion_type == "intermediate"  # only fusion type used by Ablation 1 / 4
        self.fusion = FusionToggle(d_model=d_model, num_heads=num_heads, dropout=dropout,
                                    use_cross_attn=config["use_cross_attn"])
        self.temporal = make_temporal()
        self.cls_head, self.reg_head = make_heads()
        self.gas_recon_head = nn.Sequential(nn.Linear(d_model * 2, 128), nn.ReLU(),
                                             nn.Dropout(cls_dropout), nn.Linear(128, 6))

    def forward(self, batch: dict):
        device = next(self.parameters()).device
        rgb_seq = batch["rgb_seq"].to(device); ir_seq = batch["ir_seq"].to(device)
        rgb_avail = batch["rgb_avail"].to(device); ir_avail = batch["ir_avail"].to(device)
        gas_seq = batch["gas_seq"].to(device); gas_mask = batch["gas_mask"].to(device)
        gas_delta = batch["gas_delta"].to(device); pad_mask = batch["pad_mask"].to(device)
        B, T = rgb_avail.shape

        rgb_feat = self.rgb_encoder(rgb_seq)
        ir_feat = self.ir_encoder(ir_seq)
        gas_feat = self.gas_encoder(gas_seq, gas_mask, gas_delta, pad_mask)

        rgb_proj = self.rgb_proj(rgb_feat) * rgb_avail.unsqueeze(-1).float()
        ir_proj = self.ir_proj(ir_feat) * ir_avail.unsqueeze(-1).float()
        gas_proj = self.gas_proj(gas_feat) * gas_mask.unsqueeze(-1)

        rgb_flat = rgb_proj.reshape(B * T, -1)
        ir_flat = ir_proj.reshape(B * T, -1)
        gas_flat = gas_proj.reshape(B * T, -1)
        rgb_av_flat = rgb_avail.reshape(B * T)
        ir_av_flat = ir_avail.reshape(B * T)
        gas_av_flat = gas_mask.reshape(B * T).bool()
        z_flat, _ = self.fusion(rgb_flat, ir_flat, gas_flat, rgb_av_flat, ir_av_flat, gas_av_flat)
        z_seq = z_flat.reshape(B, T, -1)
        H = self.temporal(z_seq, pad_mask)
        H_T, last_idx = gather_last_valid(H, pad_mask)
        cls_logits = self.cls_head(H_T)
        reg_output = self.reg_head(H_T)

        rgb_last = rgb_proj[torch.arange(B, device=device), last_idx]
        ir_last = ir_proj[torch.arange(B, device=device), last_idx]
        gas_recon = self.gas_recon_head(torch.cat([rgb_last, ir_last], dim=1))

        return {"cls_logits": cls_logits, "reg_output": reg_output, "gas_recon": gas_recon, "last_idx": last_idx}


class AblationModelCustomRGB(AblationModel):
    def __init__(self, config, **kwargs):
        rgb_encoder_factory = kwargs.pop("rgb_encoder_factory")
        super().__init__(config, **kwargs)
        d_model = self.d_model
        dropout = kwargs.get("dropout", 0.5)
        self.rgb_encoder = rgb_encoder_factory()
        self.rgb_proj = nn.Sequential(
            nn.Linear(self.rgb_encoder.feature_dim, d_model),
            nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout),
        )

## Threshold calibration and probability collection

In [ ]:
def calibrate_threshold(scores, labels, thresholds=None):
    if thresholds is None:
        thresholds = THRESHOLDS_TO_SWEEP
    if len(set(labels)) < 2:
        return 0.5, 0.0
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = [1 if s >= t else 0 for s in scores]
        f = f1_score(labels, preds, average="binary", zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def collect_probs_labels(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch)
            probs = torch.softmax(out["cls_logits"], dim=1).cpu().numpy()
            all_labels.extend(batch["label"].cpu().numpy().tolist())
            all_probs.extend(probs.tolist())
    return np.array(all_probs), all_labels


def get_calibrated_threshold(model, calib_loader, device):
    probs, labels = collect_probs_labels(model, calib_loader, device)
    threshold, _ = calibrate_threshold(probs[:, 1].tolist(), labels)
    return threshold


## Data: train (calibration), val (Grad-CAM/SHAP gallery), test (misclassification cases)

In [ ]:
BATCH1 = 1

train_ds = DayLevelSequenceDataset(split="train")
val_ds = DayLevelSequenceDataset(split="val", shared_pixel_cache=train_ds._pixel_cache)
test_ds = DayLevelSequenceDataset(split="test", shared_pixel_cache=train_ds._pixel_cache)

train_loader_calib = DataLoader(train_ds, batch_size=BATCH1, shuffle=False,
                                 collate_fn=day_sequence_collate, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH1, shuffle=False,
                         collate_fn=day_sequence_collate, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH1, shuffle=False,
                          collate_fn=day_sequence_collate, num_workers=0)

print(f"train (calibration only): {len(train_ds)} trajectories")
print(f"val (Grad-CAM/SHAP gallery): {len(val_ds)} trajectories")
print(f"test (misclassification cases only, read-only): {len(test_ds)} trajectories")


## Checkpoint discovery: best seed per config

In [ ]:
def find_best_seed_checkpoint(base_dir, run_subdir_pattern, seeds, threshold_field="val_f1", f1_field="val_f1"):
    best_seed, best_f1, best_ckpt, best_thr = None, -1.0, None, None
    for seed in seeds:
        run_dir = run_subdir_pattern(seed)
        ckpt_path = run_dir / "model_state.pt"
        hist_path = run_dir / "history.json"
        if not ckpt_path.exists() or not hist_path.exists():
            continue
        try:
            with open(hist_path) as f:
                history = json.load(f)
            best_epoch = max(history, key=lambda e: e.get(f1_field, -1))
            f1 = best_epoch.get(f1_field, -1)
            thr = best_epoch.get(threshold_field, best_epoch.get("threshold", 0.5))
            if f1 > best_f1:
                best_seed, best_f1, best_ckpt, best_thr = seed, f1, ckpt_path, thr
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] reading history for seed {seed} at {hist_path}: {type(e).__name__}: {e}")
            FAILURES.append({"stage": "checkpoint_discovery", "path": str(hist_path),
                              "type": type(e).__name__, "message": str(e), "traceback": tb})
    return best_seed, best_ckpt, best_thr


full_seed, full_ckpt, full_thr = find_best_seed_checkpoint(
    FUSION_OUT, lambda s: FUSION_OUT / RUN_KEY / f"seed_{s}", range(10),
    threshold_field="val_threshold", f1_field="val_f1",
)
print(f"Full Model: best seed={full_seed}, threshold={full_thr}, checkpoint={full_ckpt}")

abl1_seed, abl1_ckpt, abl1_thr = find_best_seed_checkpoint(
    ABLATION_OUT, lambda s: ABLATION_OUT / f"ablation_1_no_cbam_seed{s}" / f"seed_{s}", range(6),
    threshold_field="threshold", f1_field="val_f1",
)
print(f"Ablation 1 (no CBAM): best seed={abl1_seed}, threshold={abl1_thr}, checkpoint={abl1_ckpt}")

abl4_seed, abl4_ckpt, abl4_thr = find_best_seed_checkpoint(
    ABLATION_OUT, lambda s: ABLATION_OUT / f"ablation_4_mean_impute_not_grud_seed{s}" / f"seed_{s}", range(6),
    threshold_field="threshold", f1_field="val_f1",
)
print(f"Ablation 4 (mean-impute): best seed={abl4_seed}, threshold={abl4_thr}, checkpoint={abl4_ckpt}")

if full_ckpt is None or abl1_ckpt is None or abl4_ckpt is None:
    print("\n[WARNING] At least one required checkpoint was not found.")


In [ ]:
def load_full_model(rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0"):
    m = TrimodalFusionModel(rgb_backbone_name=rgb_backbone_name, ir_backbone_name=ir_backbone_name,
                             d_model=64, dropout=0.5, cls_dropout=0.5, freeze_visual_backbone=True).to(DEVICE)
    m.load_state_dict(torch.load(full_ckpt, map_location=DEVICE))
    m.eval()
    return m


def load_ablation_model(config_key, ckpt_path):
    m = AblationModelCustomRGB(
        ABLATION_CONFIGS[config_key], d_model=64, dropout=0.5, cls_dropout=0.5,
        freeze_visual_backbone=True,
        rgb_encoder_factory=lambda: VisualEncoderToggle("convnext_tiny", True, use_cbam=True),
    ).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.eval()
    return m


full_model = load_full_model() if full_ckpt else None
ablation1_model = load_ablation_model("ablation_1_no_cbam", abl1_ckpt) if abl1_ckpt else None
ablation4_model = load_ablation_model("ablation_4_mean_impute_not_grud", abl4_ckpt) if abl4_ckpt else None
print("Models loaded:", {"full_model": full_model is not None,
                          "ablation1_model": ablation1_model is not None,
                          "ablation4_model": ablation4_model is not None})


## 4 test-set misclassifications

In [ ]:
all_full_seeds = []
for seed in range(10):
    ckpt_path = FUSION_OUT / RUN_KEY / f"seed_{seed}" / "model_state.pt"
    if not ckpt_path.exists():
        print(f"  [SKIP] Full Model seed {seed}: checkpoint not found at {ckpt_path}")
        continue
    try:
        m = load_full_model()
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        m.eval()
        all_full_seeds.append(m)
    except Exception as e:
        tb = traceback.format_exc()
        print(f"  [FAIL] loading Full Model seed {seed}: {type(e).__name__}: {e}")
        FAILURES.append({"stage": "load_all_full_seeds", "seed": seed,
                          "type": type(e).__name__, "message": str(e), "traceback": tb})

print(f"{len(all_full_seeds)} / 10 Full Model seeds loaded for ensemble misclassification lookup.")

test_traj_meta = [{"fruit": b["fruit"][0], "label_str": b["label_str"][0]} for b in test_loader]
ensemble_test_probs = np.mean([collect_probs_labels(m, test_loader, DEVICE)[0][:, 1] for m in all_full_seeds], axis=0)
test_labels = collect_probs_labels(all_full_seeds[0], test_loader, DEVICE)[1]

ensemble_threshold = get_calibrated_threshold(all_full_seeds[0], train_loader_calib, DEVICE)
thr_per_seed = [get_calibrated_threshold(m, train_loader_calib, DEVICE) for m in all_full_seeds]
ensemble_threshold = float(np.median(thr_per_seed))

ensemble_test_preds = (ensemble_test_probs >= ensemble_threshold).astype(int).tolist()

misclassified_indices = [i for i, (p, l) in enumerate(zip(ensemble_test_preds, test_labels)) if p != l]
print(f"\nCalibrated threshold (median across {len(all_full_seeds)} seeds): {ensemble_threshold:.3f}")
print(f"Test set: {len(test_labels)} trajectories, {len(misclassified_indices)} misclassified\n")

MISCLASSIFIED_CASES = []
for i in misclassified_indices:
    kind = "false_positive" if test_labels[i] == 0 else "false_negative"
    case = {"test_index": i, "fruit": test_traj_meta[i]["fruit"], "true_label": test_traj_meta[i]["label_str"],
            "predicted": "spoiled" if ensemble_test_preds[i] == 1 else "not_spoiled",
            "prob_spoiled": float(ensemble_test_probs[i]), "kind": kind}
    MISCLASSIFIED_CASES.append(case)
    print(f"  [{kind}] test_index={i} fruit={case['fruit']} true={case['true_label']} "
          f"predicted={case['predicted']} P(spoiled)={case['prob_spoiled']:.3f}")

with open(EXPLAIN_OUT / "test_misclassifications.json", "w") as f:
    json.dump(MISCLASSIFIED_CASES, f, indent=2)


# Grad-CAM (visual encoders)

In [ ]:
def get_last_conv_layer(backbone):
    last = None
    for m in backbone.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last


def gradcam_on_frame(model, batch, modality, timestep, target_class=1):
    encoder = model.rgb_encoder if modality == "rgb" else model.ir_encoder
    seq_key = "rgb_seq" if modality == "rgb" else "ir_seq"
    target_layer = get_last_conv_layer(encoder.backbone)

    activations, gradients = {}, {}
    def fwd_hook(module, inp, out):
        activations["value"] = out
    def bwd_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0]

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    b = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in batch.items()}
    b[seq_key] = b[seq_key].clone().requires_grad_(True)

    model.zero_grad()
    out = model(b)
    target = out["cls_logits"][0, target_class]
    target.backward()

    h1.remove(); h2.remove()

    T = b[seq_key].shape[1]
    act = activations["value"].view(1, T, *activations["value"].shape[1:])[0, timestep]
    grad = gradients["value"].view(1, T, *gradients["value"].shape[1:])[0, timestep]

    weights = grad.mean(dim=(1, 2))
    cam = torch.relu((weights[:, None, None] * act).sum(dim=0))
    cam = cam / (cam.max() + 1e-8)
    cam_np = cam.detach().cpu().numpy()
    cam_resized = cv2.resize(cam_np, (IMAGE_SIZE, IMAGE_SIZE))

    frame = b[seq_key][0, timestep, :3].detach().cpu().permute(1, 2, 0).numpy()
    frame = (frame * np.array(IMAGENET_STD)) + np.array(IMAGENET_MEAN)
    frame = frame.clip(0, 1)

    return cam_resized, frame

In [ ]:
if full_model is None or ablation1_model is None:
    print("[SKIP].")
else:
    val_batches = list(val_loader)
    val_batches.sort(key=lambda b: (b["fruit"][0], b["label_str"][0]))
    n_rows = len(val_batches)

    fig, axes = plt.subplots(n_rows, 4, figsize=(15, 3.0 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    gradcam_gallery_failures = []
    for row, batch in enumerate(val_batches):
        fruit = batch["fruit"][0]; label = batch["label_str"][0]
        try:
            with torch.no_grad():
                probe_full = full_model(batch)
            last_idx_full = probe_full["last_idx"].item()
            with torch.no_grad():
                probe_abl1 = ablation1_model(batch)
            last_idx_abl1 = probe_abl1["last_idx"].item()

            cam_rgb_full, frame_rgb = gradcam_on_frame(full_model, batch, "rgb", last_idx_full, target_class=1)
            cam_rgb_abl1, _ = gradcam_on_frame(ablation1_model, batch, "rgb", last_idx_abl1, target_class=1)
            cam_ir_full, frame_ir = gradcam_on_frame(full_model, batch, "ir", last_idx_full, target_class=1)
            cam_ir_abl1, _ = gradcam_on_frame(ablation1_model, batch, "ir", last_idx_abl1, target_class=1)

            for col, (frame, cam, title) in enumerate([
                (frame_rgb, cam_rgb_full, "RGB Full Model"),
                (frame_rgb, cam_rgb_abl1, "RGB Ablation-1 (no CBAM)"),
                (frame_ir, cam_ir_full, "IR Full Model"),
                (frame_ir, cam_ir_abl1, "IR Ablation-1 (no CBAM)"),
            ]):
                ax = axes[row, col]
                ax.imshow(frame)
                ax.imshow(cam, cmap="jet", alpha=0.45)
                ax.set_title(f"{fruit} / {label}\n{title}" if col == 0 else title, fontsize=8)
                ax.axis("off")
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] Grad-CAM gallery row {row} ({fruit}/{label}): {type(e).__name__}: {e}")
            gradcam_gallery_failures.append({"row": row, "fruit": fruit, "label": label,
                                              "type": type(e).__name__, "message": str(e), "traceback": tb})
            for col in range(4):
                axes[row, col].axis("off")
                axes[row, col].set_title("FAILED", fontsize=8, color="red")

    FAILURES.extend(gradcam_gallery_failures)
    plt.tight_layout()
    plt.savefig(EXPLAIN_OUT / "gradcam_full_gallery.png", dpi=100)
    plt.show()
    print(f"\nGallery: {n_rows} val trajectories, {len(gradcam_gallery_failures)} failures. "
          f"Saved -> {EXPLAIN_OUT / 'gradcam_full_gallery.png'}")


## Grad-CAM on the 4 test-set misclassifications

In [ ]:
if full_model is None:
    print("[SKIP] Requires full_model; failed to load (see FAILURES).")
elif not MISCLASSIFIED_CASES:
    print("[INFO] No misclassifications found")
else:
    n_cases = len(MISCLASSIFIED_CASES)
    fig, axes = plt.subplots(n_cases, 4, figsize=(15, 3.2 * n_cases))
    if n_cases == 1:
        axes = axes.reshape(1, -1)

    misclass_gradcam_failures = []
    for row, case in enumerate(MISCLASSIFIED_CASES):
        try:
            batch = list(test_loader)[case["test_index"]]
            with torch.no_grad():
                probe = full_model(batch)
            last_idx = probe["last_idx"].item()

            cam_rgb, frame_rgb = gradcam_on_frame(full_model, batch, "rgb", last_idx, target_class=1)
            cam_ir, frame_ir = gradcam_on_frame(full_model, batch, "ir", last_idx, target_class=1)

            for col, (frame, cam, title) in enumerate([
                (frame_rgb, None, "RGB (original)"),
                (frame_rgb, cam_rgb, "RGB Grad-CAM"),
                (frame_ir, None, "IR (original)"),
                (frame_ir, cam_ir, "IR Grad-CAM"),
            ]):
                ax = axes[row, col]
                ax.imshow(frame)
                if cam is not None:
                    ax.imshow(cam, cmap="jet", alpha=0.45)
                row_title = (f"{case['kind'].replace('_', ' ').upper()}: {case['fruit']}, "
                             f"true={case['true_label']}, pred={case['predicted']} "
                             f"(P={case['prob_spoiled']:.2f})\n{title}") if col == 0 else title
                ax.set_title(row_title, fontsize=8)
                ax.axis("off")
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] Grad-CAM on misclassification {row} ({case}): {type(e).__name__}: {e}")
            misclass_gradcam_failures.append({"case": case, "type": type(e).__name__,
                                               "message": str(e), "traceback": tb})
            for col in range(4):
                axes[row, col].axis("off")

    FAILURES.extend(misclass_gradcam_failures)
    plt.tight_layout()
    plt.savefig(EXPLAIN_OUT / "gradcam_misclassifications.png", dpi=120)
    plt.show()
    print(f"\n{n_cases} misclassification(s) visualized, {len(misclass_gradcam_failures)} failures. "
          f"Saved -> {EXPLAIN_OUT / 'gradcam_misclassifications.png'}")


# SHAP (gas features)

In [ ]:
if not SHAP_AVAILABLE:
    raise ImportError("shap is not installed.")

GAS_FEATURE_NAMES = ["mean_ppm", "std_ppm", "left_mean", "right_mean", "|L-R|", "session_idx"]


def build_gas_background(model, loader):
    rows = []
    for batch in loader:
        with torch.no_grad():
            probe_out = model(batch)
        last_idx = probe_out["last_idx"].item()
        rows.append(batch["gas_seq"][0, last_idx].numpy())
    return np.stack(rows)


def make_predict_fn(model, fixed_batch, last_idx):
    def predict_fn(gas_matrix):
        probs = []
        for row in gas_matrix:
            b = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in fixed_batch.items()}
            b["gas_seq"][0, last_idx] = torch.tensor(row, dtype=torch.float32)
            with torch.no_grad():
                out = model(b)
            probs.append(torch.softmax(out["cls_logits"], dim=1)[0, 1].item())
        return np.array(probs)
    return predict_fn


def run_shap_over_loader(model, loader, background, label=""):
    batches = list(loader)
    shap_values_all, meta = [], []
    for i, batch in enumerate(batches):
        try:
            with torch.no_grad():
                probe_out = model(batch)
            last_idx = probe_out["last_idx"].item()
            this_gas = batch["gas_seq"][0, last_idx].numpy()

            predict_fn = make_predict_fn(model, batch, last_idx)
            explainer = shap.KernelExplainer(predict_fn, background, silent=True)
            sv = explainer.shap_values(this_gas.reshape(1, -1), nsamples=100, silent=True)
            shap_values_all.append(np.array(sv).flatten())
            meta.append({"fruit": batch.get("fruit", [None])[0], "label": batch.get("label_str", [None])[0]})
            print(f"  [{label}] trajectory {i + 1}/{len(batches)} done")
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] [{label}] trajectory {i}: {type(e).__name__}: {e}")
            FAILURES.append({"stage": f"shap_{label}", "trajectory_index": i,
                              "type": type(e).__name__, "message": str(e), "traceback": tb})

    shap_values_all = np.array(shap_values_all) if shap_values_all else np.zeros((0, 6))
    mean_abs_shap = np.abs(shap_values_all).mean(axis=0) if len(shap_values_all) else np.zeros(6)
    return shap_values_all, mean_abs_shap, meta

## SHAP over the full validation set

In [ ]:
if full_model is None or ablation4_model is None:
    print("[SKIP]")
else:
    background_full = build_gas_background(full_model, val_loader)
    background_abl4 = build_gas_background(ablation4_model, val_loader)

    shap_full, mean_abs_full, meta_full = run_shap_over_loader(full_model, val_loader, background_full, label="full_model")

    print("\nSHAP for Ablation 4 (mean-impute)")
    shap_abl4, mean_abs_abl4, meta_abl4 = run_shap_over_loader(ablation4_model, val_loader, background_abl4, label="ablation4")

    with open(EXPLAIN_OUT / "gas_shap_values.json", "w") as f:
        json.dump({
            "feature_names": GAS_FEATURE_NAMES,
            "full_model": {"shap_values": shap_full.tolist(), "mean_abs_shap": mean_abs_full.tolist(), "meta": meta_full},
            "ablation4": {"shap_values": shap_abl4.tolist(), "mean_abs_shap": mean_abs_abl4.tolist(), "meta": meta_abl4},
        }, f, indent=2)
    print(f"\nSaved -> {EXPLAIN_OUT / 'gas_shap_values.json'}")


In [ ]:
if full_model is not None and ablation4_model is not None and len(shap_full) and len(shap_abl4):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, mean_abs, title in [(axes[0], mean_abs_full, "Full Model (GRU-D)"),
                                 (axes[1], mean_abs_abl4, "Ablation 4 (mean-impute)")]:
        order = np.argsort(mean_abs)[::-1]
        ax.barh([GAS_FEATURE_NAMES[i] for i in order][::-1], mean_abs[order][::-1], color="#2ecc71")
        ax.set_xlabel("Mean |SHAP value| (impact on P(spoiled))")
        ax.set_title(title)
    plt.tight_layout()
    plt.savefig(EXPLAIN_OUT / "gas_shap_importance_comparison.png", dpi=120)
    plt.show()

    print("Full Model feature ranking (most to least influential):")
    for i in np.argsort(mean_abs_full)[::-1]:
        print(f"  {GAS_FEATURE_NAMES[i]:<14} mean|SHAP|={mean_abs_full[i]:.4f}")
    print("\nAblation 4 (mean-impute) feature ranking (most to least influential):")
    for i in np.argsort(mean_abs_abl4)[::-1]:
        print(f"  {GAS_FEATURE_NAMES[i]:<14} mean|SHAP|={mean_abs_abl4[i]:.4f}")

    top_full = GAS_FEATURE_NAMES[np.argsort(mean_abs_full)[::-1][0]]
    top_abl4 = GAS_FEATURE_NAMES[np.argsort(mean_abs_abl4)[::-1][0]]
    print(f"\nTop feature: Full Model = '{top_full}', Ablation 4 = '{top_abl4}' -- "
          f"{'SAME top feature' if top_full == top_abl4 else 'DIFFERENT top feature'} "
          f"between the two configurations.")
    print(f"Saved -> {EXPLAIN_OUT / 'gas_shap_importance_comparison.png'}")
else:
    print("[SKIP]")


In [ ]:
if full_model is None or ablation4_model is None:
    print("[SKIP]")
elif not MISCLASSIFIED_CASES:
    print("[INFO] No misclassifications found ")
else:
    misclass_shap_rows = []
    for case in MISCLASSIFIED_CASES:
        batch = list(test_loader)[case["test_index"]]
        row = {"fruit": case["fruit"], "true_label": case["true_label"],
               "predicted": case["predicted"], "kind": case["kind"]}
        for model, model_name, background in [(full_model, "full_model", background_full),
                                                (ablation4_model, "ablation4", background_abl4)]:
            try:
                with torch.no_grad():
                    probe_out = model(batch)
                last_idx = probe_out["last_idx"].item()
                this_gas = batch["gas_seq"][0, last_idx].numpy()
                predict_fn = make_predict_fn(model, batch, last_idx)
                explainer = shap.KernelExplainer(predict_fn, background, silent=True)
                sv = np.array(explainer.shap_values(this_gas.reshape(1, -1), nsamples=100, silent=True)).flatten()
                row[f"{model_name}_shap"] = sv.tolist()
                row[f"{model_name}_top_feature"] = GAS_FEATURE_NAMES[int(np.argmax(np.abs(sv)))]
            except Exception as e:
                tb = traceback.format_exc()
                print(f"  [FAIL] SHAP on misclassification case {case} ({model_name}): "
                      f"{type(e).__name__}: {e}")
                FAILURES.append({"stage": f"shap_misclassification_{model_name}", "case": case,
                                  "type": type(e).__name__, "message": str(e), "traceback": tb})
                row[f"{model_name}_shap"] = None
        misclass_shap_rows.append(row)
        print(f"  {case['kind']}: {case['fruit']} -- Full Model top feature: "
              f"{row.get('full_model_top_feature')}, Ablation 4 top feature: "
              f"{row.get('ablation4_top_feature')}")

    with open(EXPLAIN_OUT / "misclassification_shap.json", "w") as f:
        json.dump(misclass_shap_rows, f, indent=2)
    print(f"\nSaved -> {EXPLAIN_OUT / 'misclassification_shap.json'}")


In [ ]:
print("=" * 90)
print("SUMMARY")
print("=" * 90)
print(f"Grad-CAM gallery: {len(val_batches) if 'val_batches' in dir() else 'N/A'} val trajectories "
      f"x 2 modalities x 2 checkpoints (Full Model, Ablation 1)")
print(f"Grad-CAM on test misclassifications: {len(MISCLASSIFIED_CASES)} cases")
print(f"SHAP over val: Full Model and Ablation 4, {len(val_ds)} trajectories each")
print(f"SHAP on test misclassifications: {len(MISCLASSIFIED_CASES)} cases x 2 checkpoints")
print(f"\nTotal failures logged this notebook: {len(FAILURES)}")
if FAILURES:
    print("See FAILURES for full type/message/traceback of each -- nothing above was "
          "silently skipped.")
print(f"\nAll outputs saved under: {EXPLAIN_OUT}")
